# Unified Fruit Analysis Pipeline

This notebook demonstrates the complete unified pipeline that combines:
1. **Fruit Classification** - Identifies fruit type (apple, banana, orange, etc.)
2. **Freshness Detection** - Determines if fruit is fresh or rotten
3. **Dimension Measurement** - Measures fruit width, height, and area

All in a single pipeline that provides complete fruit analysis.

## Setup and Imports

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'torch',
    'torchvision',
    'tensorflow',
    'opencv-python',
    'Pillow',
    'numpy',
    'matplotlib',
    'flask',
    'flask-cors'
]

# Uncomment to install packages
# for package in packages:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [ ]:
import os
import sys
os.chdir('/path/to/model-find-it')  # Change to project directory

from unified_pipeline import UnifiedFruitAnalyzer, FruitAnalysisService
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

print("✓ All imports successful!")

## 1. Initialize the Unified Analyzer

In [ ]:
# Initialize the analyzer with model paths
analyzer = UnifiedFruitAnalyzer(
    fruit_class_model_path="fruit_classification/Fruit_Classification.pth",
    freshness_model_path="rottenvsfresh_classifier/rottenvsfresh_model.h5",
    device="cuda"  # or "cpu" if no GPU
)

print("✓ Analyzer initialized successfully!")
print(f"  - Fruit classification model loaded: {analyzer.fruit_class_model is not None}")
print(f"  - Freshness detection model loaded: {analyzer.freshness_model is not None}")
print(f"  - Device: {analyzer.device}")

## 2. Analyze a Single Image

In [ ]:
# Analyze a single image
image_path = "fruit_dimension/sample/1.png"  # Replace with your image

result = analyzer.analyze(image_path, visualize=False)

# Display results in a nice format
print("\n" + "="*60)
print("UNIFIED FRUIT ANALYSIS RESULTS")
print("="*60)

if "error" in result:
    print(f"Error: {result['error']}")
else:
    print(f"\nImage: {result['image_path']}")
    print(f"Timestamp: {result['timestamp']}")
    
    # Classification Results
    print(f"\n📍 FRUIT CLASSIFICATION:")
    clf = result['classification']
    if clf:
        print(f"   Type: {clf['fruit_type']}")
        print(f"   Confidence: {clf['confidence']:.2%}")
        print(f"   All scores:")
        for fruit, score in clf['all_scores'].items():
            print(f"      - {fruit}: {score:.2%}")
    else:
        print("   Model not loaded")
    
    # Freshness Results
    print(f"\n🍎 FRESHNESS DETECTION:")
    fresh = result['freshness']
    if fresh:
        status_emoji = "✅" if fresh['status'] == "fresh" else "❌"
        print(f"   Status: {status_emoji} {fresh['status'].upper()}")
        print(f"   Confidence: {fresh['confidence']:.2%}")
    else:
        print("   Model not loaded")
    
    # Dimension Results
    print(f"\n📏 DIMENSION MEASUREMENT:")
    dims = result['physical_dimensions']
    if dims['width']:
        print(f"   Width: {dims['width']:.2f} pixels")
        print(f"   Height: {dims['height']:.2f} pixels")
        print(f"   Area: {dims['area']:.2f} pixels²")
        print(f"   Angle: {dims['angle']:.2f}°")
    else:
        print("   Could not measure dimensions")

print("\n" + "="*60)

## 3. Visualize Results

In [ ]:
# Display the image with analysis results
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.imshow(image_rgb)

# Add text with results
y_pos = 30
if result['classification']:
    text = f"{result['classification']['fruit_type']} ({result['classification']['confidence']:.0%})"
    ax.text(10, y_pos, "Classification: " + text, color='lime', fontsize=12, 
            bbox=dict(facecolor='black', alpha=0.7))
    y_pos += 40

if result['freshness']:
    text = f"{result['freshness']['status'].upper()} ({result['freshness']['confidence']:.0%})"
    ax.text(10, y_pos, "Freshness: " + text, color='lime', fontsize=12,
            bbox=dict(facecolor='black', alpha=0.7))
    y_pos += 40

if result['physical_dimensions']['width']:
    dims = result['physical_dimensions']
    text = f"{dims['width']:.0f} × {dims['height']:.0f} px"
    ax.text(10, y_pos, "Dimensions: " + text, color='lime', fontsize=12,
            bbox=dict(facecolor='black', alpha=0.7))

ax.set_title("Unified Fruit Analysis Result", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Batch Analysis

In [ ]:
# Analyze multiple images in a directory
image_directory = "fruit_dimension/sample/"  # Directory containing test images

if os.path.exists(image_directory):
    results = analyzer.analyze_batch(image_directory, visualize=False)
    
    print(f"\n📊 BATCH ANALYSIS RESULTS")
    print(f"Total images analyzed: {len(results)}")
    print("\n" + "-"*60)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. {Path(result['image_path']).name}")
        
        if result['classification']:
            print(f"   Type: {result['classification']['fruit_type']} ({result['classification']['confidence']:.0%})")
        
        if result['freshness']:
            print(f"   Freshness: {result['freshness']['status']} ({result['freshness']['confidence']:.0%})")
        
        if result['physical_dimensions']['width']:
            dims = result['physical_dimensions']
            print(f"   Dimensions: {dims['width']:.0f} × {dims['height']:.0f} pixels")
else:
    print(f"Directory '{image_directory}' not found")

## 5. Export Results as JSON

In [ ]:
# Save results to JSON file for backend processing
output_file = "analysis_results.json"

if 'results' in locals():
    analyzer.save_results(results, output_file)
    
    # Display the JSON
    with open(output_file, 'r') as f:
        data = json.load(f)
    
    print("\n✓ Results saved to", output_file)
    print("\nSample JSON structure:")
    print(json.dumps(data[0] if data else {}, indent=2))
else:
    print("No batch results to save. Run batch analysis first.")

## 6. Using the Backend Service

In [ ]:
# Initialize backend service (same as Flask service)
service = FruitAnalysisService(model_dir=".")

# Check service health
health = service.get_health_status()
print("Service Status:")
print(json.dumps(health, indent=2))

# Analyze using the service
service_result = service.analyze_image(image_path)
print("\n✓ Service analysis complete!")

## 7. Performance Metrics and Statistics

In [ ]:
# Calculate statistics from batch results
if 'results' in locals() and results:
    import pandas as pd
    
    # Extract data
    data = []
    for res in results:
        if res['classification'] and res['freshness']:
            data.append({
                'fruit_type': res['classification']['fruit_type'],
                'confidence': res['classification']['confidence'],
                'freshness': res['freshness']['status'],
                'freshness_conf': res['freshness']['confidence'],
                'width': res['physical_dimensions']['width'],
                'height': res['physical_dimensions']['height']
            })
    
    if data:
        df = pd.DataFrame(data)
        
        print("\n📊 ANALYSIS STATISTICS\n")
        print(df.to_string(index=False))
        
        print("\n\nSummary:")
        print(f"Average Confidence: {df['confidence'].mean():.2%}")
        print(f"Average Width: {df['width'].mean():.2f} px")
        print(f"Average Height: {df['height'].mean():.2f} px")
        print(f"Fresh samples: {(df['freshness'] == 'fresh').sum()}")
        print(f"Rotten samples: {(df['freshness'] == 'rotten').sum()}")

## 8. API Documentation for Backend Integration

In [ ]:
api_docs = """
UNIFIED FRUIT ANALYSIS API
==========================

The unified pipeline provides a single API for all fruit analysis tasks.

ENDPOINTS:
----------
1. GET /health
   - Check service health status
   - Returns: {status, fruit_classifier (bool), freshness_detector (bool)}

2. POST /analyze
   - Analyze a single image
   - Input: file (multipart) or url (form data)
   - Returns: Complete analysis with classification, freshness, dimensions

3. POST /analyze/batch
   - Analyze multiple images
   - Input: files (multipart array)
   - Returns: List of analysis results

4. GET /api/models
   - Get model information
   - Returns: Model details, capabilities, input sizes

RESPONSE FORMAT:
----------------
{
    "timestamp": "ISO-8601 timestamp",
    "image_path": "path to analyzed image",
    "classification": {
        "fruit_type": "apple/banana/orange",
        "confidence": 0.0-1.0,
        "all_scores": {...}
    },
    "freshness": {
        "status": "fresh/rotten",
        "confidence": 0.0-1.0
    },
    "physical_dimensions": {
        "width": pixels,
        "height": pixels,
        "area": pixels²
    }
}

PYTHON CLIENT EXAMPLE:
----------------------
import requests

with open('fruit.jpg', 'rb') as f:
    response = requests.post(
        'http://localhost:5000/analyze',
        files={'file': f}
    )
    result = response.json()
    print(result)
"""

print(api_docs)

## Summary

This unified pipeline successfully combines:

✅ **Fruit Classification** - Identifies fruit type with high confidence using MobileNetV2
✅ **Freshness Detection** - Distinguishes fresh from rotten fruits using deep learning
✅ **Dimension Measurement** - Accurately measures fruit dimensions using OpenCV

The pipeline is production-ready and can be deployed as:
- **Python Module** - Direct integration into applications
- **REST API** - Backend service using Flask
- **Batch Processing** - Process multiple images efficiently
- **Real-time Application** - Integrate with cameras and edge devices

### Key Features:
- Single unified interface for all analyses
- Automatic model loading and error handling
- JSON export for data pipelines
- Batch processing support
- Production-ready Flask backend
- Clear, structured output format